In [1]:
from glob import glob
import pandas as pd
import os
import soundfile as sf
from tqdm import tqdm
from multiprocess import Pool
from scipy.io import wavfile
import itertools
import io
import numpy as np
import json
import re
import zipfile
from pathlib import Path

def chunks(l, n):
    for i in range(0, len(l), n):
        yield (l[i: i + n], i // n)

def multiprocessing(strings, function, cores=6, returned=True):
    df_split = chunks(strings, len(strings) // cores)
    pool = Pool(cores)
    pooled = pool.map(function, df_split)
    pool.close()
    pool.join()

    if returned:
        return list(itertools.chain(*pooled))

/usr/lib/python3/dist-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.17.3 and <1.25.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


In [2]:
from huggingface_hub import snapshot_download

snapshot_download(repo_id="mesolitica/semisupervised-audiobook", 
                  repo_type="dataset", local_dir="./semisupervised-audiobook")

/home/ubuntu/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 18 files: 100%|██████████| 18/18 [00:03<00:00,  5.60it/s]


'/home/ubuntu/semisupervised-audiobook'

In [3]:
snapshot_download(repo_id="mesolitica/nusantara-audiobook", 
                  repo_type="dataset", local_dir="./nusantara-audiobook")

Fetching 7 files: 100%|██████████| 7/7 [00:07<00:00,  1.12s/it]


'/home/ubuntu/nusantara-audiobook'

In [4]:
!ls semisupervised-audiobook

README.md			       semisupervised-audiobook-part1.json
bukan-kerana-aku-5secs-noisy.tar.gz    semisupervised-audiobook-part2.json
bukan-kerana-aku-5secs-processed.json  teme-5secs-noisy.tar.gz
bukan-kerana-aku-noisy.tar.gz	       teme-5secs-processed.json
bukan-kerana-aku-processed.json        teme-noisy.tar.gz
harry-potter-5secs-noisy.tar.gz        teme-processed.json
harry-potter-5secs-processed.json      true-case-pasentran-turki.json
harry-potter-noisy.tar.gz	       true-case-salina.json
harry-potter-processed.json


In [18]:
# import tarfile

# def loop(files):
#     files, _ = files
#     for f in tqdm(files):
#         with tarfile.open(f, "r:gz") as tar:
#             tar.extractall(path='semisupervised-audiobook')
# files = glob('semisupervised-audiobook/*.tar.gz')
# multiprocessing(files, loop, cores = len(files), returned = False)

In [21]:
# import tarfile

# def loop(files):
#     files, _ = files
#     for f in tqdm(files):
#         with tarfile.open(f, "r") as tar:
#             tar.extractall(path='nusantara-audiobook')
# files = glob('nusantara-audiobook/*.gz')
# multiprocessing(files, loop, cores = len(files), returned = False)

In [58]:
data = []
files = glob('semisupervised-audiobook/*part*.json')
for f in files:
    with open(f) as fopen:
        d = json.load(fopen)
    for k, v in d.items():
        t = v['text'].strip()
        if len(t) < 2:
            continue
        k = os.path.join('semisupervised-audiobook', k)
        if not os.path.exists(k):
            continue
        data.append({
            'audio_filename': k,
            'text': v['text'].strip(),
            'speaker': 'malay_semisupervised_audiobook'
        })
len(data), data[-1]

(8443,
 {'audio_filename': 'semisupervised-audiobook/home/husein/ssd2/audiobook/harry-potter-noisy/148.wav',
  'text': 'Dobby ini elf rumah. Mengabdi kepada sebuah rumah dan sebuah keluarga buat selama-lamanya. Mereka tahu kamu datang ke sini?',
  'speaker': 'malay_semisupervised_audiobook'})

In [59]:
files = glob('nusantara-audiobook/*.json')
for f in files:
    with open(f) as fopen:
        d = json.load(fopen)
    for k, v in d.items():
        speaker = k.split('/')[0]
        t = v['text'].strip()
        if len(t) < 2:
            continue
        k = os.path.join('nusantara-audiobook', k)
        if not os.path.exists(k):
            continue
        data.append({
            'audio_filename': k,
            'text': t,
            'speaker': f'nusantara_audiobook_{speaker}',
        })
len(data), data[-1]

(32811,
 {'audio_filename': 'nusantara-audiobook/turki/output-wav-turki/janji-tinggal-janji-2.mp3-261.wav',
  'text': 'Sebuah mandat Amerika ke atas Palestine telah dibincangkan secara ringkas, tetapi segera ditinggalkan. Kemudian, Lloyd George,',
  'speaker': 'nusantara_audiobook_turki'})

In [60]:
len(data)

32811

In [64]:
def loop(rows):
    rows, _ = rows
    data = []
    for row in tqdm(rows):
        f = row['audio_filename']
        base = f.split('/')[0] + '_audio'
        os.makedirs(base, exist_ok=True)

        audio_filename = os.path.join(base, f.replace('/', '_')).replace('.wav', '.mp3')
        speaker = row['speaker']
        t = row['text'].strip()
        if len(t) < 2:
            continue
        
        audio_np, sr = sf.read(f)
        if audio_np.ndim > 1:
            audio_np = audio_np.mean(axis=1)
        if audio_np.shape[0] < 10000:
            continue
        sf.write(audio_filename, audio_np, sr)
        
        data.append({
            'audio_filename': audio_filename,
            'text': t,
            'speaker': speaker
        })
        
    return data
        

In [65]:
processed = loop((data[:10], 0))

100%|██████████| 10/10 [00:00<00:00, 11.50it/s]


In [ ]:
processed = multiprocessing(data, loop, cores = 20)

 19%|█▉        | 312/1640 [00:31<02:11, 10.12it/s]

In [86]:
from datasets import Dataset

dataset = Dataset.from_list(processed)
dataset[0]

{'audio_filename': 'semisupervised-audiobook_audio/semisupervised-audiobook_home_husein_ssd2_audiobook_harry-potter-noisy_343.mp3',
 'text': 'Cahaya matahari memancar dari celah pokok. Kita akan mendarat, kata Fred. Dengan sedikit hentakan, kereta itu mencecah bumi.',
 'speaker': 'malay_semisupervised_audiobook'}

In [87]:
dataset.push_to_hub('malaysia-ai/Multilingual-TTS', 'malay-audiobook')

Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 61.10ba/s]
Processing Files (0 / 0): |          |  0.00B /  0.00B            
Processing Files (1 / 1): 100%|██████████| 2.31MB / 2.31MB, 5.76MB/s  
New Data Upload: |          |  0.00B /  0.00B,  0.00B/s  
Uploading the dataset shards: 100%|██████████| 1/1 [00:00<00:00,  1.16 shards/s]
No files have been modified since last commit. Skipping to prevent empty commit.


CommitInfo(commit_url='https://huggingface.co/datasets/malaysia-ai/Multilingual-TTS/commit/b4e820b3c32c27a5ee6123d31bb83cb815446478', commit_message='Upload dataset', commit_description='', oid='b4e820b3c32c27a5ee6123d31bb83cb815446478', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/malaysia-ai/Multilingual-TTS', endpoint='https://huggingface.co', repo_type='dataset', repo_id='malaysia-ai/Multilingual-TTS'), pr_revision=None, pr_num=None)

In [89]:
audio_files = [d['audio_filename'] for d in processed]

with open('malay-audiobook-audio.json', 'w') as fopen:
    json.dump(list(set(audio_files)), fopen)

In [79]:
# !zip -rq semisupervised-audiobook_audio.zip semisupervised-audiobook_audio
# !zip -rq nusantara-audiobook_audio.zip nusantara-audiobook_audio

In [80]:
# !hf upload malaysia-ai/Multilingual-TTS semisupervised-audiobook_audio.zip --repo-type=dataset

In [81]:
# !hf upload malaysia-ai/Multilingual-TTS nusantara-audiobook_audio.zip --repo-type=dataset

In [95]:
# !zip -rq semisupervised-audiobook_audio_neucodec.zip semisupervised-audiobook_neucodec
# !zip -rq nusantara-audiobook_audio_neucodec.zip nusantara-audiobook_audio_neucodec

In [96]:
# !hf upload malaysia-ai/Multilingual-TTS semisupervised-audiobook_audio_neucodec.zip --repo-type=dataset
# !hf upload malaysia-ai/Multilingual-TTS nusantara-audiobook_audio_neucodec.zip --repo-type=dataset